# kaiming-uniform-sf-init composite — cx30: Kaiming-uniform initialised weight, wrapped as nn.Parameter

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `kaiming-uniform-sf-init`, `nn-parameter-wrap`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "kaiming-uniform-sf-init"
DD_ATOM_IDS = ["kaiming-uniform-sf-init", "nn-parameter-wrap"]
DD_SUBTOPICS = ["Init: Kaiming uniform SF init", "PyTorch: nn.Parameter"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

PyTorch's `nn.Linear` and `nn.Conv2d` both initialise their weight tensor via **Kaiming uniform** (scaled by `1/sqrt(fan_in)` with a gain matching the assumed downstream nonlinearity). When you write a custom layer, you have to do this yourself: create a tensor of the right shape, fill it with kaiming-uniform samples, then **wrap it as `nn.Parameter`** so the optimizer sees it.

**The two atoms.**
- **kaiming-uniform-sf-init** — the FILL. For a weight of shape `(out, in)` (Linear) or `(out_c, in_c, kH, kW)` (Conv2d), `fan_in = in` (Linear) or `in_c * kH * kW` (Conv2d). The 'self-fan' style ARENA uses is `bound = 1 / sqrt(fan_in); weight.uniform_(-bound, bound)`. (`nn.init.kaiming_uniform_` with `a=sqrt(5)` matches PyTorch's exact convention for Linear; for clarity ARENA writes the formula by hand.)
- **nn-parameter-wrap** — `nn.Parameter(tensor)` flags the tensor as 'this is a learned param'. After wrapping, the tensor appears in `model.parameters()` and the optimizer's update rule applies to it. A raw tensor in `self.x = tensor` is NOT trained.

**Anatomy.**
```python
class MyLinear(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        # Atom A: kaiming-uniform-sf-init.
        bound = 1.0 / math.sqrt(in_f)
        w = t.empty(out_f, in_f).uniform_(-bound, bound)
        # Atom B: nn-parameter-wrap.
        self.weight = nn.Parameter(w)
    def forward(self, x):
        return x @ self.weight.T
```

**Why both atoms together.** A correctly-shaped weight tensor is useless if the optimizer can't find it; a correctly-wrapped `nn.Parameter` is broken if its init scale is wrong (too big → activations explode; too small → vanishing grads on first batch).

### Composite Exercise — Kaiming-uniform initialised weight, wrapped as nn.Parameter

**Atoms exercised together**: `kaiming-uniform-sf-init`, `nn-parameter-wrap`

Implement two helpers.

1. `cx30_init_kaiming_uniform(shape, fan_in)` — return a fresh `t.Tensor` of the given `shape`, filled with samples from `Uniform(-bound, +bound)` where `bound = 1 / sqrt(fan_in)`. Do NOT wrap as Parameter — just return the raw tensor.

2. `cx30_make_linear()` — return the class `MyLinear(nn.Module)` such that `MyLinear(in_features, out_features)`:
   - calls `super().__init__()`
   - creates `weight` via `cx30_init_kaiming_uniform((out_features, in_features), in_features)`
   - **wraps it as `nn.Parameter`** and stores as `self.weight`
   - implements `forward(self, x)` as `x @ self.weight.T`

The test checks: (a) the helper produces a tensor in the right `[-bound, +bound]` range and stats consistent with a uniform; (b) `MyLinear.weight` is an `nn.Parameter` (not just a Tensor) and shows up in `.parameters()`; (c) gradients flow through `self.weight` after one backward pass; (d) the init scale obeys `1/sqrt(fan_in)` for different `in_features`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx30_init_kaiming_uniform(shape, fan_in):
    """Return a tensor of `shape` filled with U(-1/sqrt(fan_in), +1/sqrt(fan_in))."""
    raise NotImplementedError

def cx30_make_linear():
    """Return the MyLinear class."""
    raise NotImplementedError

def _test_cx30():
    import math

    # Case A: helper produces correctly-shaped tensor with correct bound.
    t.manual_seed(0)
    w = cx30_init_kaiming_uniform((20, 100), fan_in=100)
    assert isinstance(w, t.Tensor), f'helper must return a Tensor, got {type(w).__name__}'
    assert tuple(w.shape) == (20, 100)
    bound = 1.0 / math.sqrt(100)  # = 0.1
    assert w.min().item() >= -bound - 1e-6 and w.max().item() <= bound + 1e-6, (
        f'values must lie in [-{bound}, +{bound}]; got [{w.min().item()}, {w.max().item()}]'
    )
    # Statistical sanity: uniform [-bound, +bound] has var = bound^2 / 3.
    expected_var = bound ** 2 / 3.0
    observed_var = w.var().item()
    assert abs(observed_var - expected_var) / expected_var < 0.1, (
        f'variance {observed_var:.5f} is far from uniform-bound expected {expected_var:.5f}'
    )

    # Case B: scale changes with fan_in.
    t.manual_seed(1)
    w_small = cx30_init_kaiming_uniform((50, 4),   fan_in=4)
    w_large = cx30_init_kaiming_uniform((50, 400), fan_in=400)
    # Smaller fan_in -> larger bound -> larger std.
    assert w_small.std().item() > w_large.std().item() * 5, (
        'std with fan_in=4 should be much larger than with fan_in=400 — scale rule broken'
    )

    # Case C: MyLinear wraps weight as nn.Parameter.
    MyLinear = cx30_make_linear()
    assert issubclass(MyLinear, nn.Module)
    t.manual_seed(2)
    lin = MyLinear(8, 16)
    assert isinstance(lin.weight, nn.Parameter), (
        f'lin.weight must be nn.Parameter (not raw Tensor); got {type(lin.weight).__name__}'
    )
    params = list(lin.parameters())
    assert any(p is lin.weight for p in params), 'lin.weight must appear in lin.parameters()'
    assert tuple(lin.weight.shape) == (16, 8)

    # Case D: forward + backward — gradient must flow into lin.weight.
    x = t.randn(4, 8)
    y = lin(x)
    assert tuple(y.shape) == (4, 16)
    loss = y.pow(2).sum()
    loss.backward()
    assert lin.weight.grad is not None, 'no grad on lin.weight — was it wrapped as nn.Parameter?'
    assert lin.weight.grad.abs().sum().item() > 0, 'grad is zero — backward did not reach lin.weight'

    # Case E: init scale obeys 1/sqrt(in_features) for MyLinear itself.
    t.manual_seed(3)
    lin2 = MyLinear(400, 50)
    lin3 = MyLinear(4,   50)
    # Same out_features so std is comparable on the same-shape tensors after.
    assert lin3.weight.std().item() > lin2.weight.std().item() * 5, (
        'MyLinear must apply the 1/sqrt(in_features) scale to its self.weight init'
    )
    _dd_passed.add('cx30')

_test_cx30()

<details><summary>Show solution — cx30</summary>

```python
import math

def cx30_init_kaiming_uniform(shape, fan_in):
    # Atom A (kaiming-uniform-sf-init): bound = 1/sqrt(fan_in), then in-place uniform fill.
    bound = 1.0 / math.sqrt(fan_in)
    return t.empty(*shape).uniform_(-bound, bound)

def cx30_make_linear():
    class MyLinear(nn.Module):
        def __init__(self, in_features, out_features):
            super().__init__()
            w = cx30_init_kaiming_uniform((out_features, in_features), fan_in=in_features)
            # Atom B (nn-parameter-wrap): without this, the optimizer never sees `weight`.
            self.weight = nn.Parameter(w)

        def forward(self, x):
            return x @ self.weight.T

    return MyLinear
```

Two failure modes the test catches: (1) returning a raw tensor instead of `nn.Parameter` — forward still works, but `lin.parameters()` is empty and the optimizer silently fails to train the weight. (2) Wrong init scale — e.g. `randn` instead of bounded uniform — produces values an order of magnitude too large for `fan_in=100`, blowing up the first forward pass through a deep stack. The `1/sqrt(fan_in)` bound is what `nn.Linear` uses by default, derived from the 'preserve activation variance' argument in He et al. 2015.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx30'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx30',
        'subtopics': ["Init: Kaiming uniform SF init", "PyTorch: nn.Parameter"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()